# Question Answering with Transformers
Dataset: SQuAD v1.1 (Kaggle - `stanfordu/stanford-question-answering-dataset`)

Building an **extractive question-answering** system: given a passage (context) and a question, the model extracts the exact answer span from the passage, using pretrained transformer models from Hugging Face.

## Download the dataset from Kaggle

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("stanfordu/stanford-question-answering-dataset")

print("Path to dataset files:", path)

## Install & Import libraries
Installing Hugging Face `transformers` (and `torch` as its backend) to load pretrained question-answering pipelines, and checking whether a GPU is available (strongly recommended for this task — transformer inference on CPU is much slower). We pin `transformers==4.46.3` since the latest release has a known bug where the `"question-answering"` pipeline task is missing from its task registry.

In [ ]:
!pip install -q "transformers==4.46.3" torch

import json
import glob
import os
import re
import string
import random
from collections import Counter

import pandas as pd
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
print("Using GPU" if device == 0 else "Using CPU (this will be slow — enable a GPU runtime if possible)")

## Load and parse the SQuAD v1.1 dataset
SQuAD's JSON structure is nested: `data -> [articles] -> paragraphs -> [{context, qas: [{question, id, answers: [{text, answer_start}]}]}]`. We flatten `dev-v1.1.json` (the standard evaluation split) into a simple table of `(id, context, question, answer_text, answer_start)` — one row per question.

In [ ]:
json_files = glob.glob(os.path.join(path, "**", "*.json"), recursive=True)
print("Found JSON files:", json_files)

dev_path = [f for f in json_files if 'dev' in os.path.basename(f).lower()][0]
train_path = [f for f in json_files if 'train' in os.path.basename(f).lower()][0]

def flatten_squad(filepath):
    with open(filepath, encoding='utf-8') as f:
        squad = json.load(f)

    rows = []
    for article in squad['data']:
        for paragraph in article['paragraphs']:
            context = paragraph['context']
            for qa in paragraph['qas']:
                # SQuAD v1.1 always has at least one answer per question
                answer = qa['answers'][0]
                rows.append({
                    'id': qa['id'],
                    'context': context,
                    'question': qa['question'],
                    'answer_text': answer['text'],
                    'answer_start': answer['answer_start']
                })
    return pd.DataFrame(rows)

dev_df = flatten_squad(dev_path)
print("Dev set questions:", dev_df.shape[0])
dev_df.head()

## Load a pretrained question-answering model
Using Hugging Face's `pipeline("question-answering")` with `distilbert-base-cased-distilled-squad` — DistilBERT, already fine-tuned on SQuAD. No training needed: the pipeline takes a `(question, context)` pair and directly returns the predicted answer span.

In [ ]:
qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad",
    device=device
)

# Quick look at a few predictions
for i in range(3):
    row = dev_df.iloc[i]
    result = qa_pipeline(question=row['question'], context=row['context'])
    print("QUESTION :", row['question'])
    print("GOLD     :", row['answer_text'])
    print("PREDICTED:", result['answer'], f"(score: {result['score']:.3f})")
    print()

## SQuAD evaluation metrics: Exact Match & F1
Implementing the standard SQuAD evaluation functions:
- **Exact Match (EM)**: 1 if the predicted answer exactly matches the gold answer after normalization (lowercasing, removing punctuation/articles/extra whitespace), else 0.
- **F1**: token-level overlap between the predicted and gold answer, treating them as bags of words — more forgiving of partial matches or extra/missing words.

In [ ]:
def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text):
        return ' '.join(text.split())
    def remove_punc(text):
        return ''.join(ch for ch in text if ch not in string.punctuation)
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def exact_match_score(prediction, ground_truth):
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))

def f1_score_qa(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(ground_truth).split()
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return int(pred_tokens == gold_tokens)
    if num_same == 0:
        return 0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

## Evaluate the pretrained model
Running the QA pipeline on a sample of the dev set (transformer inference is slow question-by-question, especially on CPU — increase `N_SAMPLES` for a more thorough evaluation once you've confirmed everything works).

In [ ]:
random.seed(42)
N_SAMPLES = 300
sample_df = dev_df.sample(n=min(N_SAMPLES, len(dev_df)), random_state=42).reset_index(drop=True)

def evaluate_model(qa_pipeline, df):
    em_total, f1_total = 0, 0
    for _, row in df.iterrows():
        result = qa_pipeline(question=row['question'], context=row['context'])
        em_total += exact_match_score(result['answer'], row['answer_text'])
        f1_total += f1_score_qa(result['answer'], row['answer_text'])
    n = len(df)
    return em_total / n * 100, f1_total / n * 100

em, f1 = evaluate_model(qa_pipeline, sample_df)
print(f"DistilBERT (distilbert-base-cased-distilled-squad) -> EM: {em:.2f} | F1: {f1:.2f}  (n={len(sample_df)})")

## Compare different base models
Comparing several transformer architectures fine-tuned for SQuAD-style QA: DistilBERT (already loaded above), BERT (large), and RoBERTa. Each has a different architecture and training recipe, so accuracy and speed both vary.

In [ ]:
model_names = {
    'DistilBERT': 'distilbert-base-cased-distilled-squad',
    'BERT (large)': 'bert-large-uncased-whole-word-masking-finetuned-squad',
    'RoBERTa': 'deepset/roberta-base-squad2',
}

results = {'DistilBERT': (em, f1)}  # reuse the result computed above

for name, model_id in model_names.items():
    if name == 'DistilBERT':
        continue
    print(f"Loading {name} ({model_id})...")
    pipe = pipeline("question-answering", model=model_id, device=device)
    em_i, f1_i = evaluate_model(pipe, sample_df)
    results[name] = (em_i, f1_i)
    print(f"{name} -> EM: {em_i:.2f} | F1: {f1_i:.2f}")

print("\n=== Model Comparison ===")
for name, (em_i, f1_i) in results.items():
    print(f"{name:15s} -> EM: {em_i:.2f} | F1: {f1_i:.2f}")

## Simple interactive Q&A function
A lightweight command-line-style interface directly in the notebook: pass any passage and question, get an answer back. (A standalone Streamlit app version is provided separately — see `qa_app.py`.)

In [ ]:
def ask(context, question, model_pipeline=qa_pipeline):
    result = model_pipeline(question=question, context=context)
    print(f"Q: {question}")
    print(f"A: {result['answer']}  (confidence: {result['score']:.3f})")

# Example usage
example_context = (
    "The Amazon rainforest is a moist broadleaf tropical rainforest in the Amazon biome "
    "that covers most of the Amazon basin of South America. This basin encompasses 7 million "
    "square kilometers, of which 5.5 million square kilometers are covered by the rainforest."
)
ask(example_context, "How large is the Amazon basin?")
ask(example_context, "What type of forest is the Amazon rainforest?")